# 02 — Topic centroids and firm-by-topic continuous scores

**Purpose.** The heart of the deterministic rail: represent every Stage-3
topic as the embedding centroid of its actual evidence sentences, score every
(filing, topic) pair by top-$k$ mean cosine over the filing's sentences, and
keep the top-$k$ sentences themselves — the by-product that is the evidence
the LLM layer reads, the audit page for any score, and the sampling frame for
the mechanism audit. Also embeds the Stage-2 hearing summaries, which notebook
04 needs for the pair-level decomposition.

**Unit symmetry.** Topic centroids are built from *sentences* (stance-record
evidence, segmented by the same pinned segmenter as Item 1) and compared
against *sentences* — the plan's like-compared-with-like defense (§3).

**Input contracts** (asserted below; produced by the final Stage-3 chain —
codebook v1.0 frozen sha `fd81e781…`, notebooks 05 and 10):

| File | Required columns |
|---|---|
| `interim/topics/taxonomy/codebook_v1.parquet` | topic_id, theme_id, label, theme_label, definition |
| `interim/topics/taxonomy/track2_assigned.parquet` | packageId, turn_idx, sentences (1-based indices), topic_id, sim |
| `interim/topics/track2/member_turns.parquet` | packageId, turn_idx, sentences (texts) |
| `interim/relevance/item1/sentences.parquet` + `emb_*.npy` | from notebook 01, row-aligned |
| `data/cleaned/01_text_summarization/hearing_summaries.parquet` | packageId, summary (from Stage-2 `04`) |

Evidence text is materialized here by joining the placed stance records
(placement floor $\tau = 0.68$, the rule ratified at the codebook freeze —
records below the floor are "other" and carry no topic) to the member-turn
sentence lists; record indices are 1-based, the same convention the Stage-3
check notebook (06) reads with.

**Outputs**, under `interim/relevance/scores/`:

| File | Contents |
|---|---|
| `topic_index.parquet` + `topic_centroids_{enc}.npy` | topic order and centroid matrix, row-aligned |
| `firm_topic_scores.parquet` | one row per (filing, topic): top-$k$ mean score |
| `evidence_sentences.parquet` | the top-$k$ sentences per (filing, topic), ranked, with cosines |
| `summary_emb_{enc}.npy` + `summary_index.parquet` | hearing-summary embeddings for 04 |

**Gate (plan §8).** Every topic has enough evidence sentences to carry a
centroid (floor printed), length-decile diagnostics for the score, and a
spot-check showing top-matching sentences that read as genuinely on-topic.

## Setup and contract checks

Every upstream file is asserted before any computation — a missing or
schema-drifted input fails here, on the first cell, with the file named.

In [1]:
import os
import sys
import time

sys.path.insert(0, "../lib")

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import relevance as R
from paths import DATA_REPO, HF_CACHE, TOPICS_INTERIM, relevance_run_dir

os.environ["HF_HOME"] = str(HF_CACHE)

ENCODER_NAME = os.environ.get("ENCODER_NAME", R.ENCODER)
ENC_SHORT = ENCODER_NAME.split("/")[-1]
TOP_K = 5                                    # plan §3: top-k mean, frozen at dev
MIN_CENTROID_SENTENCES = 20                  # a topic below this floor is flagged
TAU_PLACEMENT = 0.68                         # Stage-3 ratified placement floor (notebook 10)

ITEM1 = relevance_run_dir("item1")
OUT = relevance_run_dir("scores")

CODEBOOK = TOPICS_INTERIM / "taxonomy" / "codebook_v1.parquet"
ASSIGNED = TOPICS_INTERIM / "taxonomy" / "track2_assigned.parquet"
TURNS = TOPICS_INTERIM / "track2" / "member_turns.parquet"
SUMMARIES = DATA_REPO / "cleaned" / "01_text_summarization" / "hearing_summaries.parquet"

for path, cols in [
    (CODEBOOK, {"topic_id", "theme_id", "label", "theme_label", "definition"}),
    (ASSIGNED, {"packageId", "turn_idx", "sentences", "topic_id", "sim"}),
    (TURNS, {"packageId", "turn_idx", "sentences"}),
    (ITEM1 / "sentences.parquet", {"accessionNo", "cik", "gvkey", "fyear",
                                    "sent_idx", "sentence"}),
    (SUMMARIES, {"packageId", "summary"}),
]:
    assert path.exists(), f"missing input: {path}"
    have = set(pq.read_schema(path).names)
    assert cols <= have, f"{path.name}: missing columns {cols - have}"
assert (ITEM1 / f"emb_{ENC_SHORT}.npy").exists(), "run 01 (embed) first"
print("all input contracts satisfied")
print(f"encoder: {ENCODER_NAME}   top-k: {TOP_K}   placement floor: {TAU_PLACEMENT}")

all input contracts satisfied
encoder: BAAI/bge-large-en-v1.5   top-k: 5   placement floor: 0.68


## Step 1 — Topic centroids from evidence sentences

A centroid built from the definition text alone would represent the codebook's
prose style; built from the stance records' evidence sentences it represents
how the topic is actually spoken about, which is what the Item 1 sentences
must be matched against.

The evidence text is materialized here: placed records (sim ≥ τ) joined to
their member-turn sentence lists, each 1-based cited index resolved to its
sentence, one sentence = one evidence unit (the like-for-like unit the scores
compare), deduplicated within topic so a sentence cited by several records
does not multiply its weight. Topics below the evidence floor keep a centroid
(definition + what evidence exists) but carry a flag that follows them into
every downstream table.

In [2]:
import torch
from transformers import AutoModel, AutoTokenizer

codebook = pd.read_parquet(CODEBOOK)

assigned = pd.read_parquet(
    ASSIGNED, columns=["packageId", "turn_idx", "sentences", "topic_id", "sim"])
placed = assigned.loc[assigned["sim"] >= TAU_PLACEMENT].copy()

turns = pd.read_parquet(TURNS, columns=["packageId", "turn_idx", "sentences"])
turns = turns.rename(columns={"sentences": "turn_sentences"})
ev = placed.merge(turns, on=["packageId", "turn_idx"], how="left", validate="m:1")
assert ev["turn_sentences"].notna().all(), "record without a member turn"

# 1-based cited indices -> sentence texts (same convention as Stage-3 nb 06)
ev["evidence"] = [
    [ts[i - 1] for i in idxs if 1 <= i <= len(ts)]
    for idxs, ts in zip(ev["sentences"], ev["turn_sentences"])
]
ev = ev[["topic_id", "evidence"]].explode("evidence").dropna(subset=["evidence"])
n_before = len(ev)
ev = ev.drop_duplicates(["topic_id", "evidence"])
print(f"placed records : {len(placed):,} of {len(assigned):,} (tau {TAU_PLACEMENT})")
print(f"evidence units : {len(ev):,} sentences "
      f"({n_before - len(ev):,} within-topic duplicates dropped)")

tokenizer = AutoTokenizer.from_pretrained(ENCODER_NAME)
encoder = AutoModel.from_pretrained(ENCODER_NAME, dtype=torch.float16,
                                    device_map="cuda")
encoder.eval()

topic_rows, centroids = [], []
for topic in codebook.itertuples():
    evid = ev.loc[ev["topic_id"] == topic.topic_id, "evidence"]
    texts = [topic.definition] + evid.tolist()
    vecs = R.encode(texts, encoder, tokenizer, ENCODER_NAME).astype(np.float32)
    centroid = vecs.mean(axis=0)
    centroid /= np.linalg.norm(centroid)
    centroids.append(centroid.astype(np.float16))
    topic_rows.append({"topic_id": topic.topic_id, "theme_id": topic.theme_id,
                       "label": topic.label, "n_evidence": len(evid),
                       "thin_centroid": len(evid) < MIN_CENTROID_SENTENCES})

topic_index = pd.DataFrame(topic_rows)
centroid_mat = np.stack(centroids)
topic_index.to_parquet(OUT / "topic_index.parquet", index=False)
np.save(OUT / f"topic_centroids_{ENC_SHORT}.npy", centroid_mat)

print(f"topics: {len(topic_index)}  "
      f"(thin: {int(topic_index['thin_centroid'].sum())} "
      f"below {MIN_CENTROID_SENTENCES} evidence sentences)")
print(f"evidence per topic: median {topic_index['n_evidence'].median():.0f}, "
      f"min {topic_index['n_evidence'].min()}")

placed records : 29,979 of 32,385 (tau 0.68)
evidence units : 37,104 sentences (556 within-topic duplicates dropped)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

topics: 111  (thin: 0 below 20 evidence sentences)
evidence per topic: median 263, min 27


## Step 2 — Firm-by-topic scores with evidence

For each filing: its sentence block times the centroid matrix, then the
top-$k$ mean per topic and the identities of those $k$ sentences. One filing
at a time keeps memory flat; results land in preallocated arrays (a
(filings × topics × $k$) evidence tensor, not fourteen million Python dicts)
and become tables in one shot at the end.

In [3]:
sentences = pd.read_parquet(ITEM1 / "sentences.parquet")
emb = np.load(ITEM1 / f"emb_{ENC_SHORT}.npy", mmap_mode="r")
assert len(sentences) == len(emb), "01's alignment contract violated"
cent32 = centroid_mat.astype(np.float32).T          # (dim, n_topics)
n_topics = cent32.shape[1]
topic_ids = topic_index["topic_id"].to_numpy()

groups = sentences.groupby("accessionNo", sort=False).indices
n_groups = len(groups)
cik_arr = sentences["cik"].to_numpy()
gvkey_arr = sentences["gvkey"].to_numpy()
fy_arr = sentences["fyear"].to_numpy()
sentidx_arr = sentences["sent_idx"].to_numpy()

acc_list, cik_list, gvkey_list, fy_list = [], [], [], []
score_mat = np.empty((n_groups, n_topics), dtype=np.float32)
nsent_arr = np.empty(n_groups, dtype=np.int32)
ev_row = np.full((n_groups, n_topics, TOP_K), -1, dtype=np.int64)   # global row index
ev_cos = np.full((n_groups, n_topics, TOP_K), np.nan, dtype=np.float32)

started = time.time()
for g, (acc, idx) in enumerate(groups.items()):
    block = emb[idx].astype(np.float32)              # (n_sent, dim)
    sims = block @ cent32                            # (n_sent, n_topics)
    k = min(TOP_K, len(idx))
    top = np.argpartition(-sims, k - 1, axis=0)[:k]  # (k, n_topics), unordered
    vals = np.take_along_axis(sims, top, axis=0)
    order = np.argsort(-vals, axis=0)                # rank within the k
    top = np.take_along_axis(top, order, axis=0)
    vals = np.take_along_axis(vals, order, axis=0)

    score_mat[g] = vals.mean(axis=0)
    nsent_arr[g] = len(idx)
    ev_row[g, :, :k] = idx[top].T
    ev_cos[g, :, :k] = vals.T
    acc_list.append(acc)
    cik_list.append(cik_arr[idx[0]])
    gvkey_list.append(gvkey_arr[idx[0]])
    fy_list.append(int(fy_arr[idx[0]]))
    if (g + 1) % 5000 == 0:
        print(f"  {g + 1:,}/{n_groups:,} filings  "
              f"({(time.time() - started) / 60:.1f} min)")

acc_arr = np.array(acc_list, dtype=object)
scores = pd.DataFrame({
    "accessionNo": np.repeat(acc_arr, n_topics),
    "cik": np.repeat(np.array(cik_list, dtype=object), n_topics),
    "gvkey": np.repeat(np.array(gvkey_list, dtype=object), n_topics),
    "fyear": np.repeat(np.array(fy_list), n_topics),
    "topic_id": np.tile(topic_ids, n_groups),
    "score": score_mat.ravel(),
    "n_sentences": np.repeat(nsent_arr, n_topics),
})

mask = ev_row >= 0
g_idx, t_idx, r_idx = np.nonzero(mask)
evidence = pd.DataFrame({
    "accessionNo": acc_arr[g_idx],
    "topic_id": topic_ids[t_idx],
    "rank": r_idx.astype(np.int16),
    "sent_idx": sentidx_arr[ev_row[mask]],
    "cosine": ev_cos[mask],
})

scores.to_parquet(OUT / "firm_topic_scores.parquet", compression="zstd", index=False)
evidence.to_parquet(OUT / "evidence_sentences.parquet", compression="zstd", index=False)
print(f"\nscores   : {len(scores):,} (filings x topics)")
print(f"evidence : {len(evidence):,} rows")

  5,000/21,966 filings  (0.2 min)


  10,000/21,966 filings  (0.3 min)


  15,000/21,966 filings  (0.5 min)


  20,000/21,966 filings  (0.7 min)



scores   : 2,438,226 (filings x topics)
evidence : 12,190,686 rows


## Step 3 — Length-decile diagnostic

The plan's named pitfall: a score that rises mechanically with document
length is a size-correlated measurement error headed into firm regressions.
Top-$k$ mean should be far flatter than max across length deciles — this
table is the check, and if it fails, the per-decile standardization switch in
notebook 04 turns on.

In [4]:
scores["len_decile"] = pd.qcut(scores["n_sentences"], 10, labels=False,
                                duplicates="drop")
decile_mean = scores.groupby("len_decile")["score"].mean()
print("mean top-k score by filing-length decile:")
print(decile_mean.round(4).to_string())
spread = decile_mean.max() - decile_mean.min()
print(f"\ndecile spread: {spread:.4f} "
      f"({spread / scores['score'].std():.2f} sd) — "
      "flag for per-decile standardization if > 0.5 sd")

mean top-k score by filing-length decile:
len_decile
0    0.6200
1    0.6393
2    0.6464
3    0.6521
4    0.6555
5    0.6604
6    0.6642
7    0.6694
8    0.6738
9    0.6807

decile spread: 0.0607 (1.74 sd) — flag for per-decile standardization if > 0.5 sd


## Step 4 — Spot-check: do the top matches read as on-topic?

Three topics, their three highest-scoring filings, the best-matching sentence
of each. This is the qualitative gate — a high cosine attached to an
off-topic sentence here means the centroid or the filters are wrong, and no
downstream statistic will say so as plainly.

In [5]:
sent_lookup = sentences.set_index(["accessionNo", "sent_idx"])["sentence"]
for topic_id in topic_index.loc[~topic_index["thin_centroid"], "topic_id"].head(3):
    label = topic_index.set_index("topic_id").loc[topic_id, "label"]
    best = scores.loc[scores["topic_id"] == topic_id].nlargest(3, "score")
    print("=" * 78)
    print(f"{topic_id}  {label}")
    for row in best.itertuples():
        ev = evidence.loc[(evidence["accessionNo"] == row.accessionNo)
                          & (evidence["topic_id"] == topic_id)
                          & (evidence["rank"] == 0)].iloc[0]
        text = sent_lookup.loc[(row.accessionNo, ev["sent_idx"])]
        print(f"  {row.gvkey} fy{row.fyear}  score {row.score:.3f}")
        print(f"    {text[:110]}")

3  NASA Authorization Acts


  008972 fy1999  score 0.737
    The funding of government programs is subject to congressional appropriations.
  008972 fy2000  score 0.730
    The funding of Government programs is subject to congressional appropriations.


  008972 fy2001  score 0.730
    The funding of Government programs is subject to congressional appropriations.
4  International Space Station


  126721 fy2004  score 0.737
    The centerpiece of our current NASA activities is the International Space Station.
  126721 fy2003  score 0.731
    The centerpiece of our current NASA activities is the International Space Station.


  126721 fy2002  score 0.721
    Our Systems Group plays a key role in the International Space Station (ISS), one of the most complex scientifi
6  National Polar-orbiting Operational Environmental Satellite System


  060900 fy2012  score 0.774
    We are evaluating our options for these satellites and depending on our eventual use of these satellites, we m
  060900 fy2004  score 0.760
    Certain Other Risks to Our Satellites .


  060900 fy2013  score 0.759
    These requirements may place constraints on available capacity on our satellites for other programming.


## Step 5 — Hearing-summary embeddings (for the pair decomposition)

Notebook 04's pair-level residual needs each hearing's final summary as a
vector under the same encoder. Computed here while the encoder is loaded;
written row-aligned to its own index file, the same contract as the sentence
matrix.

In [6]:
summaries = pd.read_parquet(SUMMARIES, columns=["packageId", "summary"])
summaries = summaries.dropna(subset=["summary"])
summaries = summaries.loc[summaries["summary"].str.strip() != ""].reset_index(drop=True)

sum_emb = R.encode(summaries["summary"].tolist(), encoder, tokenizer, ENCODER_NAME)
np.save(OUT / f"summary_emb_{ENC_SHORT}.npy", sum_emb)
summaries[["packageId"]].to_parquet(OUT / "summary_index.parquet", index=False)
print(f"embedded {len(summaries):,} hearing summaries")

embedded 2,352 hearing summaries


## Verification

In [7]:
back = pd.read_parquet(OUT / "firm_topic_scores.parquet")
assert not back.duplicated(["accessionNo", "topic_id"]).any()
assert back["score"].between(-1.001, 1.001).all(), "cosine out of range"
print(f"1) one score per (filing, topic): {len(back):,} rows, all in [-1, 1]")

ev_back = pd.read_parquet(OUT / "evidence_sentences.parquet")
per_pair = ev_back.groupby(["accessionNo", "topic_id"]).size()
print(f"2) evidence rows per (filing, topic): min {per_pair.min()}, "
      f"max {per_pair.max()} (cap {TOP_K})")

si = pd.read_parquet(OUT / "summary_index.parquet")
sv = np.load(OUT / f"summary_emb_{ENC_SHORT}.npy", mmap_mode="r")
assert len(si) == len(sv)
print(f"3) summary embeddings aligned: {len(si):,}")

print(f"4) encoder {ENCODER_NAME}, top-k {TOP_K}, "
      f"centroid floor {MIN_CENTROID_SENTENCES}")
print("\nDone. Next: 04 (pairing, decomposition, screening calibration, regimes).")

1) one score per (filing, topic): 2,438,226 rows, all in [-1, 1]


2) evidence rows per (filing, topic): min 1, max 5 (cap 5)
3) summary embeddings aligned: 2,352
4) encoder BAAI/bge-large-en-v1.5, top-k 5, centroid floor 20

Done. Next: 04 (pairing, decomposition, screening calibration, regimes).
